# Lab 9: Image Classification

**Duration:** 2 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
In Lab 6 the program saw a picture as numbers. Now the program **understands** the picture. An **image classifier** looks at one whole picture and answers one question: *what is this a picture of?* It gives a list of guesses and a score for each. In this lab you run a pretrained classifier on sample pictures, on your own camera pictures and on live video, and you learn why a classifier is right for some jobs and wrong for others.

**By the end of this lab you can:**
1. Explain what image classification does and what its output is (a list of classes with scores).
2. Run a pretrained YOLO classifier on a picture and read the top-5 guesses.
3. Explain what a score (confidence) means and why a score is not a promise.
4. Show with an experiment that a classifier gives one answer for the whole picture, even when the picture has many objects.
5. Run the classifier on live video and measure how often it is right.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **No training in this lab.** The model was trained by Ultralytics on a very large set of labelled pictures. You only use it. Training your own model comes much later in the course.

## 2. Background

**Classification, one answer per picture.** A classifier takes a picture and returns a **label** for the whole picture, for example `espresso` or `school bus`. It does not say *where* the object is, and it does not count objects. If the picture has a cup, a keyboard and a mouse, the classifier still gives one list of guesses for the whole picture.

**Classes.** The classifier knows a fixed list of names, its **classes**. This model knows **1000 classes** from a famous collection called ImageNet (animals, tools, food, vehicles and many more). It can only choose from this list. If your object is not in the list, the model picks the closest class, and it is often wrong.

**Score (confidence).** For every class the model gives a number between 0 and 1. All 1000 numbers add up to 1. The biggest number is the model's best guess. A score of 0.92 means *the model is very sure*. It does **not** mean it is right 92 times out of 100. Models can be sure and wrong.

| Word | Meaning |
|---|---|
| Top-1 | The single best guess. It is correct when it is the true class. |
| Top-5 | The five best guesses. It is correct when the true class is anywhere in the five. |

**Why top-5?** Many classes look alike (for example 120 dog breeds). The true class is often second or third. Top-5 accuracy is always higher than top-1 accuracy.

**Where classification is used.** Sorting photos, checking if a product on a line is good or bad, finding out if a plant leaf is sick, deciding if a room is empty or full. It works best when the picture has **one main subject**.

**The next labs.** Detection (Lab 10) answers *what and where*. Oriented detection (Lab 11), segmentation (Lab 12), pose (Lab 13) and tracking (Lab 14) build on it.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(9)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (15 min)

### Step 1: A Helper to Show Pictures
Run the cell. It makes a small function called `show` that displays picture files under a cell.

In [ ]:
from IPython.display import Image, display

def show(*names, width=360):
    """Show picture files under the cell."""
    for n in names:
        print(n)
        display(Image(filename=n, width=width))

### Step 2: Your Workspace
This notebook lives in the `lab9` folder inside your own folder. Everything this lab makes is saved here. Check where you are:

In [ ]:
%%bash
pwd
ls

### Step 3: Check PyTorch and YOLO
If `CUDA available` is `False` or the PyTorch version ends with `+cpu`, stop and call your instructor. Do **not** run `pip install` in this lab.

In [ ]:
import torch
import ultralytics

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
print("Ultralytics version:", ultralytics.__version__)

### Step 4: Get the Sample Pictures and Models
The YOLO package comes with two sample pictures. This copies them into the folder of this notebook. It also copies the model files that the Lab 0 setup downloaded, so nothing is downloaded during the lab.

In [ ]:
import glob, os, shutil
import ultralytics

assets = os.path.join(os.path.dirname(ultralytics.__file__), "assets")
for f in glob.glob(assets + "/*.jpg"):
    shutil.copy(f, ".")

# Model files downloaded by the Lab 0 setup
models = os.path.join(os.environ["LAB_HOME"], "models")
for f in glob.glob(models + "/*.pt"):
    shutil.copy(f, ".")

print(sorted(glob.glob("*.jpg")))
print(sorted(glob.glob("*.pt")))
show("bus.jpg", "zidane.jpg")

---
# Part B: Classify Pictures (35 min)

### Step 1: Classify the Sample Pictures
The classifier gives **one list** of guesses for the whole picture. The first run takes longer because the model loads and the GPU prepares.

In [ ]:
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
cls_model = YOLO("yolo26n-cls.pt")

def classify(image):
    result = cls_model(image, device=device, verbose=False)[0]
    print(f"Image: {image}   Image size: {result.orig_shape}")
    print(f"Model knows {len(result.names)} classes")
    print("Top 5 guesses:")
    for rank, i in enumerate(result.probs.top5, start=1):
        print(f"  {rank}. {result.names[i]:<20} {float(result.probs.data[i]) * 100:5.1f} %")
    print(f"Inference time: {result.speed['inference']:.1f} ms")

classify("bus.jpg")
print()
classify("zidane.jpg")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What is the top-1 guess for bus.jpg, and its score?:
- What is the top-1 guess for zidane.jpg, and its score? Is it a good description of the picture?:
- The picture zidane.jpg shows people. Why is there no class called `person` in the top 5?:

### Step 2: Test Your Own Picture
Use the picture from Lab 6 (`snapshot.jpg`, saved by the Lab 6 notebook). Or put any picture in this folder and change the name.

In [ ]:
import os

snap = "../lab6/snapshot.jpg"      # the Lab 6 notebook saves its pictures in your lab6 folder
if os.path.exists(snap):
    classify(snap)
    show(snap)
else:
    print("The picture snapshot.jpg does not exist yet.")
    print("It is the photo your USB camera takes in the Lab 6 notebook, cell Part B, Step 3.")
    print("Go back to Lab 6, run that cell, then come back here.")
    print("Or skip it: copy any .jpg picture into this folder and use its name instead.")

### Step 3: One Answer for Many Objects
Take a picture with **several** objects in it. The cell below classifies the whole picture, and then classifies **crops** of the left half and the right half of the same picture. Compare the three answers.

In [ ]:
import cv2

def classify_crop(image, name):
    img = cv2.imread(image)
    h, w = img.shape[:2]
    parts = {"whole": img, "left half": img[:, : w // 2], "right half": img[:, w // 2 :]}
    for label, part in parts.items():
        cv2.imwrite(name, part)
        r = cls_model(name, device=device, verbose=False)[0]
        i = int(r.probs.top1)
        print(f"{label:<11} -> {r.names[i]:<22} {float(r.probs.top1conf) * 100:5.1f} %")

classify_crop("bus.jpg", "part.jpg")
print()
classify_crop("zidane.jpg", "part.jpg")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Are the three answers (whole, left, right) the same or different?:
- What does this tell you about a classifier and a picture with many objects?:

### Step 4: An Object That Is Not in the 1000 Classes
Hold an object that is probably not in the list (for example a pen cap, a USB stick or your ID card) in front of the camera and take a picture with the Lab 6 notebook, or use any picture you have. Classify it. Write what the model says and its score.

In [ ]:
import os
mine = "../lab6/snapshot.jpg"
if os.path.exists(mine):
    classify(mine)
    show(mine)
else:
    print("No snapshot.jpg yet. Take a picture in the Lab 6 notebook, or copy any .jpg here and change the name.")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Object you tried::
- What did the model say, and with what score?:
- Is the model sure and wrong, or unsure? What does this teach you about scores?:

---
# Part C: Live Classification on the USB Camera (30 min)

### Step 1: Run the Live Classifier
Run the cell. The live picture appears under it with the top-3 guesses written on the picture. Hold different objects in front of the camera. Press **Stop** when you are done, so the camera and the memory are free.

The FPS includes sending every picture to your browser, so it is lower than the model-only speed.

In [ ]:
import cv2
import labview

def process(frame, state):
    r = cls_model(frame, device=device, verbose=False)[0]
    top3 = r.probs.top5[:3]
    for row, i in enumerate(top3):
        text = f"{r.names[i]}  {float(r.probs.data[i]) * 100:4.0f} %"
        cv2.putText(frame, text, (10, 30 + 30 * row), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(frame, f"{state['fps']:.0f} FPS", (10, frame.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return frame

live = labview.live(process)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which object gave the highest score? What was it and what score?:
- Which object was classified wrongly? What did the model say instead?:
- FPS shown on the screen::

---
# Part D: Measure How Good It Is (30 min)

### Step 1: A Test Table
Test **five** different objects (different, not the same type). For each object run the live cell again (or take a picture), hold the object so it fills the picture, and write the top-1 guess. Mark top-1 correct if the first guess is right, and top-5 correct if the right answer is in the five guesses (you can see the top-5 in Part B).

| Object | Top-1 guess and score | Top-1 correct? | In top-5? |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |

Double-click the table to fill it in. Then calculate: top-1 accuracy = correct / 5. Top-5 accuracy = in top-5 / 5.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Top-1 accuracy (correct out of 5)::
- Top-5 accuracy (correct out of 5)::
- Which object was the hardest, and why do you think so?:

> **Where next.** A classifier tells you *what*. The next lab uses a **detector**, which tells you *what and where*, and can find many objects in one picture.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The classifier runs on `bus.jpg` and `zidane.jpg` and the top-5 table is shown | |
| 2 | Your own picture is classified, and you explain the result | |
| 3 | The whole / left / right experiment is done and you explain the difference | |
| 4 | The live classifier runs and the top-3 guesses and the FPS show on the screen | |
| 5 | The test table of Part D is complete, with top-1 and top-5 accuracy calculated | |
| 6 | All answers are written in the answer cells | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. What does an image classifier output? Why is it called one answer per picture?:
- 2. What is the difference between top-1 and top-5? Why is top-5 accuracy never lower than top-1?:
- 3. The score of a guess is 0.95 and the guess is wrong. How can this happen?:
- 4. A picture has a cup and a laptop. What does a classifier answer, and what can it not tell you?:
- 5. The model knows 1000 classes. What happens when you show it an object that is not in the list?:
- 6. Name one real job where a classifier is a good choice, and one where you need a detector instead.:
- 7. Why is the FPS on the screen lower than the model-only inference time?:
- 8. How would you make the test in Part D fairer? Name two changes.:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `CUDA available: False` | The PyTorch of your environment is the CPU version | Stop and call your instructor. Do not run `pip install`. |
| `yolo26n-cls.pt` not found | The Lab 0 setup did not download the model files | Ask your teacher. The setup step of Lab 0 downloads the model files once. |
| `Cannot open the camera` | Another program uses the camera, or the cable is loose | Press Stop in other live cells, shut down other kernels, ask your teacher to check the camera on the teacher page |
| No picture appears under the live cell | Cells above were not run, or an earlier error | Run the cells from the top. Read the error message first. |
| Every object gives a strange answer | The object is not one of the 1000 classes, or it is small in the picture | Hold the object closer so it fills the picture. Try another object. |
| `CUDA out of memory` | Other notebooks use the shared 8 GB | Press Stop, shut down other kernels (Kernel, Shut Down All Kernels) and run again |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(9)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Ultralytics, Image Classification: https://docs.ultralytics.com/tasks/classify/
- ImageNet (the 1000 classes): https://www.image-net.org/